# ใบงานนักเรียน: สร้าง RAG Chatbot ด้วย BGE-M3 และ Ollama 🤖

ชื่อ ____________________ ชั้น ______ เลขที่ ______

กติกา:

- อ่านโจทย์แล้วพิมพ์โค้ดในเซลล์ `TODO`
- รันเซลล์จากบนลงล่าง
- ตรวจผลในแต่ละ Checkpoint ก่อนทำขั้นต่อไป
- หากติดขัด ให้ดูคำใบ้ก่อนเปิดไฟล์ Solution

## เป้าหมายการเรียนรู้

เมื่อจบบทเรียน นักเรียนจะสามารถ:

1. อธิบายขั้นตอนของ RAG ได้
2. ทำความสะอาดและแบ่งเอกสารเป็น chunks
3. ใช้ BGE-M3 เปลี่ยนข้อความเป็น embedding
4. ใช้ cosine similarity ค้นหาข้อมูลที่เกี่ยวข้อง
5. ส่ง Context และคำถามให้ Qwen ผ่าน OpenAI-compatible API
6. สร้าง RAG Chatbot ที่จำบทสนทนาได้

## 1) รู้จัก RAG

RAG ย่อมาจาก **Retrieval-Augmented Generation** เป็นการค้นข้อมูลที่เกี่ยวข้องก่อน แล้วส่งข้อมูลนั้นให้โมเดลช่วยสร้างคำตอบ

```text
เอกสาร → ทำความสะอาด → แบ่ง Chunk → Embedding → เก็บ Vector
คำถาม → Embedding → ค้นหา Chunk → สร้าง Context → Qwen → คำตอบ
```

คำถาม: เพราะเหตุใดเราจึงไม่ส่งเอกสารทั้งหมดให้โมเดลทุกครั้ง?

## 2) เตรียมโปรแกรมและโมเดล

ก่อนเริ่ม ให้เปิด Ollama และติดตั้งโมเดลใน Terminal:

```bash
ollama pull bge-m3
ollama pull qwen3.5:4b
```

จากนั้นติดตั้ง Python packages ที่จำเป็นในเซลล์ด้านล่าง

In [3]:
# TODO: ติดตั้ง openai, requests, numpy และ scikit-learn ด้วย %pip install

from openai import OpenAI
OLLAMA_BASE_URL = "http://localhost:11434/v1/"
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",  # Ollama ต้องการค่า แต่จะไม่ตรวจสอบ key นี้
)

## 3) ตรวจการเชื่อมต่อ Ollama

ใช้ `requests.get()` เรียก `http://localhost:11434/api/tags` แล้วแสดงชื่อโมเดลที่พบ

คำใบ้: เรียก `r.raise_for_status()` ก่อนอ่าน `r.json()`

In [4]:
# TODO: import requests และตรวจว่า Ollama เชื่อมต่อได้

OLLAMA_HOST = "http://localhost:11434"

### Checkpoint 1

- [ ] เชื่อมต่อ Ollama สำเร็จ
- [ ] พบโมเดล `bge-m3`
- [ ] พบโมเดล `qwen3.5:4b`

## 4) สร้างเอกสาร

สร้าง list ชื่อ `documents` ที่มีข้อความอย่างน้อย 3 ประโยคเกี่ยวกับประเทศไทย แล้วแสดงจำนวนเอกสาร

In [5]:
# TODO: สร้าง documents อย่างน้อย 3 ประโยค
documents = [
    "เควินชอบเล่น roblox ",
    "น้องพีท ชอบกิน pizza ",
    "poom เก่งคณิต",
    "note นั่งสมาธิเก่ง",
    "    "
]


## 5) ทำความสะอาดข้อความ

สร้าง `clean_docs` โดย:

1. วนอ่านข้อความแต่ละรายการ
2. ใช้ `.strip()` ลบช่องว่างหัวท้าย
3. เก็บเฉพาะข้อความที่ไม่ว่าง

In [6]:
# TODO: ทำความสะอาด documents
clean_docs = []
print("Before cleaning Document : ", len(documents))
for document in documents:
    document = document.strip()
    
    if document:
        clean_docs.append(document)
print( "Document : ",len(clean_docs))

Before cleaning Document :  5
Document :  4


## 6) แบ่งข้อความเป็น Chunk

แบ่งเอกสารแต่ละรายการเป็นส่วนละ 100 ตัวอักษร แล้วเก็บใน `chunks`

คำใบ้:

```python
for i in range(0, len(document), 100):
    ...
```

In [7]:
# TODO: สร้าง chunks จาก clean_docs
chunks = []

for document in clean_docs:
    for i in range(0,len(document),100):
        chunk = document[i:i + 100]
        chunks.append(chunk)
print("Chunks : ", len(chunks))

Chunks :  4


### Checkpoint 2

- [ ] `clean_docs` ไม่มีข้อความว่าง
- [ ] `chunks` มีอย่างน้อย 3 รายการ
- [ ] แต่ละ chunk ยาวไม่เกิน 100 ตัวอักษร

## 7) สร้าง Embedding ด้วย BGE-M3

เขียนฟังก์ชัน `embed(text)` เพื่อส่งข้อความไปยัง `/api/embed`

ข้อมูล JSON ที่ต้องส่ง:

```python
{"model": "bge-m3", "input": text}
```

ฟังก์ชันต้องคืนค่า `r.json()["embeddings"][0]`

In [9]:
# TODO: เขียนฟังก์ชัน embed(text)
def embed(text):
    response = client.embeddings.create(
        model= "bge-m3",
        input = text
    )
    results = []
    for item in response.data:
        results.append(item.embedding)
    return results

embeddings = embed(chunks)
print("Embeddings. : ",len(embeddings), " Dimension : ", len(embeddings[0]))
print("Vector : ", embeddings[1])

Embeddings. :  4  Dimension :  1024
Vector :  [-0.05911681428551674, 0.0041891830042004585, -0.01846967823803425, 0.01597742922604084, -0.021948929876089096, 0.020973511040210724, 0.0014892126200720668, -0.025935711339116096, 0.01674330234527588, -0.005708975717425346, 0.019912712275981903, 0.02276313491165638, -0.017693959176540375, -0.016772238537669182, 0.04868840426206589, -0.01880555972456932, -0.0167380403727293, -0.0013136998750269413, 0.0021675957832485437, -0.021919002756476402, -0.008803445845842361, 0.011576986871659756, -0.005665990523993969, 0.017870232462882996, -0.026359517127275467, 0.025026671588420868, -0.0005758698680438101, -0.005308960098773241, 0.025520972907543182, -0.026610342785716057, 0.03973107784986496, -0.019235452637076378, 0.012256819754838943, -0.06587829440832138, -0.03400902450084686, -0.03136540949344635, -0.018993554636836052, -0.07232058048248291, -0.023154284805059433, -0.018558073788881302, -0.005277440417557955, -0.010371877811849117, 0.031335566

## 8) สร้าง Vector Store

เรียก `embed()` กับทุก chunk แล้วเก็บผลใน `embeddings` จากนั้นจับคู่ `chunks` และ `embeddings` ด้วย `zip()`

In [10]:
# TODO: สร้าง embeddings และ vector_store
# TODO: สร้าง documents อย่างน้อย 3 ประโยค
# documents = [
#     "เควินชอบเล่น roblox ",
#     "น้องพีท ชอบกิน pizza ",
#     "poom เก่งคณิต",
#     "note นั่งสมาธิเก่ง",
#     "    ",
# ]


vector_store = list(zip(chunks,embeddings))
print(vector_store)

[('เควินชอบเล่น roblox', [-0.03676719218492508, -0.013046088628470898, -0.02395131252706051, -0.020597226917743683, -0.024974476546049118, 0.02554609440267086, 0.039573099464178085, -0.05129096284508705, 0.015185019001364708, -0.005780728068202734, 0.020982734858989716, 0.008296046406030655, -0.032589953392744064, 0.013557259924709797, 0.0460084043443203, 0.019520223140716553, -0.0017664082115516067, -0.01599101535975933, 0.02115507796406746, -0.013767368160188198, -0.014717782847583294, -0.010435059666633606, 0.024375109001994133, 0.04691683128476143, -0.0070717972703278065, 0.04186633601784706, 0.04284115135669708, -0.019152270630002022, -0.012405927293002605, -0.021766643971204758, -0.02481299266219139, 0.03782124072313309, -0.03173980489373207, -0.035539865493774414, -0.02515595220029354, -0.04817298799753189, 0.020954320207238197, 0.01864694431424141, -0.01853882148861885, -0.008986740373075008, -0.042570337653160095, 0.001065556425601244, 0.031044941395521164, -0.0246657598763704

## 9) รับคำถามและค้นหา Top 3

กำหนดคำถามใน `query` แล้วทำตามลำดับ:

1. สร้าง `query_vec`
2. คำนวณ `cosine_similarity` ระหว่างคำถามกับ embeddings
3. เรียงคะแนนจากมากไปน้อย
4. เก็บ index 3 อันดับแรกใน `top`
5. แสดงคะแนนและ chunk

In [12]:
# TODO: import numpy และ cosine_similarity แล้วค้นหา Top 3
query = "เควินชอบเล่นเกมอะไร"
query_vector = embed(query)
print(query_vector)

[[-0.03582114726305008, -0.03689006343483925, -0.0542147196829319, -0.00473934318870306, -0.0037021583411842585, 0.02266804873943329, 0.026276135817170143, -0.050878651440143585, 0.002620245097205043, 0.009559868834912777, 0.031998809427022934, -0.002923039486631751, -0.030698681250214577, 0.01345620397478342, 0.026541227474808693, -0.018018784001469612, 0.008532661013305187, -0.03517572581768036, 0.0023767047096043825, -0.014900687150657177, 0.006247107405215502, -0.013538138009607792, 0.06633595377206802, 0.062161799520254135, 0.011397717520594597, 0.04339765012264252, 0.013041547499597073, -0.022777486592531204, -0.014905337244272232, -0.03070661425590515, 0.0021362064871937037, 0.007216566242277622, -0.017484279349446297, -0.05519663915038109, -0.028315365314483643, -0.014168472029268742, 0.023551026359200478, -0.004869319032877684, -0.03174535930156708, -0.009730568155646324, -0.000973490416072309, 0.006075134500861168, 0.027175530791282654, 0.01425913255661726, 0.0253226328641176

### Checkpoint 3

ดูผลการค้นหาแล้วตอบ:

1. Chunk อันดับแรกเกี่ยวข้องกับคำถามหรือไม่?
2. คะแนนของอันดับแรกเป็นเท่าใด?
3. เพราะเหตุใดคะแนนมากจึงควรอยู่ก่อนคะแนนน้อย?

## 10) สร้าง Context

นำ chunks จาก index ใน `top` มาต่อกันด้วย `\n` แล้วสร้าง prompt รูปแบบนี้:

```text
Context:
ข้อมูลที่ค้นพบ

Question: คำถามของผู้ใช้
```

In [18]:
# TODO: สร้าง context และ prompt แล้ว print ดูผล

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

scores = cosine_similarity(query_vector, embeddings)[0]
print(scores)
top = np.argsort(scores)[::-1][:3]
max_score = 0
print(top)
for i in top:
    print(scores[i],chunks[i])
    if scores[i] > max_score:
        context = chunks[i]
        max_score = scores[i]
print("Choose context : ", context)


[0.83703604 0.48426614 0.44698319 0.39525989]
[0 1 2]
0.8370360428280385 เควินชอบเล่น roblox
0.4842661370875529 น้องพีท ชอบกิน pizza
0.44698319028923306 poom เก่งคณิต
Choose context :  เควินชอบเล่น roblox


## 11) ส่ง Prompt ให้ Qwen

ใช้ OpenAI Python SDK เชื่อมกับ Ollama แบบ OpenAI-compatible:

- `base_url` คือ `http://localhost:11434/v1/`
- `api_key` ใช้ข้อความ `ollama`
- `MODEL` คือ `qwen3.5:4b`

กำหนด system message ให้ตอบจาก Context เท่านั้น และบอกว่าไม่พบข้อมูลเมื่อ Context ไม่มีคำตอบ

In [22]:
# TODO: import OpenAI, สร้าง client และส่ง prompt ให้ qwen3.5:4b
# TODO: อ่าน response.choices[0].message.content แล้วแสดงคำตอบ

MODEL = "qwen3.5:4b"
prompt = f" Context : {context} Question : {query} "

response = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role" : "system",
        "content": "ตอบคำถามที่ได้จาก Context ที่ให้มาเท่านั้น เป็นภาษษไทยสั้นๆ"
    },
    {
        "role": "user",
        "content" : prompt
    }],
    reasoning_effort= "none",
    max_tokens= 150,
    temperature= 0.2
)
# print(response)
answer = response.choices[0].message.content

print("Question : ", query)
print("Answer : ", answer)

Question :  เควินชอบเล่นเกมอะไร
Answer :  เควินชอบเกม Roblox ครับ


### Checkpoint 4

- [ ] Qwen ตอบคำถามเป็นภาษาไทย
- [ ] คำตอบสอดคล้องกับ Context
- [ ] โค้ดใช้ OpenAI SDK แต่โมเดลยังทำงานใน Ollama

คำถาม: เหตุใด `api_key="ollama"` จึงไม่ใช่ OpenAI API key จริง?

# 12) Mini Project: RAG Chatbot ที่มีความจำ 🚀

ภารกิจ:

1. เพิ่มเอกสารของตนเองอย่างน้อย 3 ประโยค
2. สร้าง chunks และ embeddings ใหม่
3. เขียน `retrieve(question, k=3)`
4. สร้าง `chat_history` ที่เริ่มด้วย system message
5. เขียน `chat(message)` เพื่อค้น Context ก่อนถาม Qwen
6. ทดสอบถามต่อเนื่องอย่างน้อย 2 คำถาม

In [ ]:
# Test
# Add document

# embeddings
# Cosine simmilarity
# Chat (history)



In [ ]:
# TODO: เพิ่มเอกสาร แล้วสร้าง clean_docs, chunks และ embeddings ใหม่
project_documents = []

## 12.1) เขียนฟังก์ชันค้นหา

`retrieve(question, k=3)` ต้องคืน list ของผลลัพธ์ โดยแต่ละรายการมี `text` และ `score`

In [ ]:
# TODO: เขียนฟังก์ชันค้นหา chunk ที่เกี่ยวข้อง
def retrieve(question, k=3):
    pass

## 12.2) เพิ่มความจำ

สร้าง `chat_history` เป็น list และใส่ system message หนึ่งรายการ เพื่อกำหนดให้ Bot ตอบจาก Context เท่านั้น

In [ ]:
# TODO: สร้าง chat_history
chat_history = []

## 12.3) เขียนฟังก์ชัน `chat()`

ลำดับการทำงาน:

1. เรียก `retrieve(message)`
2. รวมผลการค้นหาเป็น Context
3. สร้าง user prompt ที่มี Context และ Question
4. ส่ง messages ให้ `client.chat.completions.create()`
5. เก็บข้อความผู้ใช้และคำตอบไว้ใน `chat_history`
6. คืนค่า `answer`

In [ ]:
# TODO: เขียน RAG chat ที่ค้นเอกสารและจำบทสนทนา
def chat(message):
    pass

## 12.4) ทดสอบ Chatbot

ถามอย่างน้อย 2 คำถาม โดยคำถามที่สองอ้างอิงเรื่องจากคำถามแรก

In [ ]:
# TODO: เรียก chat() สองครั้งและแสดงคำตอบ
# print("Bot:", chat("คำถามแรก"))
# print("Bot:", chat("คำถามต่อเนื่อง"))

## Challenge: ล้างประวัติ

เขียน `clear_chat()` เพื่อลบประวัติเดิมและใส่ system message กลับเข้าไป

In [ ]:
# TODO: เขียน clear_chat()
def clear_chat():
    pass

## งานที่ต้องส่ง

- [ ] เอกสารอย่างน้อย 6 ประโยค
- [ ] ฟังก์ชัน `embed()` ทำงาน
- [ ] แสดงผล Top 3 พร้อมคะแนน
- [ ] Qwen ตอบจาก Context
- [ ] ฟังก์ชัน `chat()` ถามต่อเนื่องได้
- [ ] ทดลองคำถามที่ไม่มีคำตอบในเอกสาร

## สะท้อนการเรียนรู้

1. Embedding แตกต่างจากข้อความธรรมดาอย่างไร?
2. Cosine similarity ช่วยใน RAG อย่างไร?
3. หาก chunk ใหญ่หรือเล็กเกินไปจะเกิดอะไรขึ้น?
4. RAG ช่วยลดการตอบจากข้อมูลที่โมเดลคิดขึ้นเองได้อย่างไร?